# Notebook Complementar — Otimização, LightGBM e Monitoramento de Manutenção Preditiva

Este notebook complementa o projeto principal com foco em:
- métricas mais adequadas para eventos raros de falha (`recall` e `F1-score`);
- teste de modelos ensemble: Random Forest, Gradient Boosting, LightGBM e XGBoost opcional;
- comparação de três configurações do LightGBM usadas no notebook principal;
- criação de pipeline com `ColumnTransformer` e SMOTE;
- salvamento do melhor pipeline com `joblib`;
- preparação para dashboard de monitoramento.

**Regra crítica de governança:** a variável `falha_maquina` é usada apenas como alvo (`y`), nunca como preditora (`X`).  
As colunas `falha_twf`, `falha_hdf`, `falha_pwf`, `falha_osf` e `falha_rnf` também são removidas das variáveis de entrada porque representam causas históricas da falha e gerariam vazamento de dados.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

## 1. Leitura local da base CSV

A base é carregada exclusivamente de um arquivo local. O notebook procura primeiro em `data/manutencao_preditiva.csv` e depois na mesma pasta em que está sendo executado.

Não há autenticação, API, download ou integração com o Kaggle.


In [2]:
# Caminhos locais aceitos, em ordem de prioridade.
CANDIDATOS_DATA_PATH = [
    Path("data") / "manutencao_preditiva.csv",
    Path("manutencao_preditiva.csv"),
]

DATA_PATH = next(
    (caminho for caminho in CANDIDATOS_DATA_PATH if caminho.exists()),
    None
)

if DATA_PATH is None:
    caminhos_testados = "\n".join(
        f"- {caminho.resolve()}" for caminho in CANDIDATOS_DATA_PATH
    )
    raise FileNotFoundError(
        "Arquivo CSV local não encontrado. Coloque "
        "'manutencao_preditiva.csv' em um destes locais:\n"
        f"{caminhos_testados}"
    )

df = pd.read_csv(DATA_PATH)

print(f"Base local carregada de: {DATA_PATH.resolve()}")
print(f"Dimensões: {df.shape[0]} linhas e {df.shape[1]} colunas")
df.head()


Base local carregada de: C:\Users\User\Documents\Projeto_Estudos\projeto_manutencao_preditiva\data\manutencao_preditiva.csv
Dimensões: 10000 linhas e 14 colunas


,udi,id_produto,tipo,temperatura_ar_k,temperatura_processo_k,velocidade_rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha_maquina,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf
0,1,M14860,M,298.1,308.6,1551.0,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408.0,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498.0,49.4,5,0,0,0,0,0,0
3,4,L47183,L,NaN,NaN,NaN,NaN,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408.0,40.0,9,0,0,0,0,0,0


## 2. Limpeza mínima e engenharia de atributos

A limpeza abaixo mantém a lógica do notebook principal e cria duas variáveis numéricas úteis para o modelo.

In [3]:
df = df.drop_duplicates().copy()

for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(exclude="number").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

df["potencia_operacional"] = df["velocidade_rotacao_rpm"] * df["torque_nm"]
df["delta_temperatura"] = df["temperatura_processo_k"] - df["temperatura_ar_k"]

df.shape

(10000, 16)

## 3. Separação segura entre X e y

Aqui está o ponto mais importante: `falha_maquina` fica apenas no alvo. As colunas de motivos técnicos de falha também são excluídas para impedir que o modelo aprenda respostas prontas do histórico.

In [4]:
TARGET = "falha_maquina"

COLUNAS_VAZAMENTO = [
    "falha_maquina",
    "falha_twf",
    "falha_hdf",
    "falha_pwf",
    "falha_osf",
    "falha_rnf"
]

COLUNAS_IDENTIFICACAO = ["udi", "id_produto"]

X = df.drop(columns=[c for c in COLUNAS_VAZAMENTO + COLUNAS_IDENTIFICACAO if c in df.columns])
y = df[TARGET]

print("Colunas usadas no modelo:")
print(X.columns.tolist())

print("\nA coluna falha_maquina está em X?", "falha_maquina" in X.columns)

Colunas usadas no modelo:
['tipo', 'temperatura_ar_k', 'temperatura_processo_k', 'velocidade_rotacao_rpm', 'torque_nm', 'desgaste_ferramenta_min', 'potencia_operacional', 'delta_temperatura']

A coluna falha_maquina está em X? False


## 4. Divisão treino/teste com estratificação

A estratificação preserva a proporção de falhas em treino e teste.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("\nProporção da classe alvo no treino:")
print(y_train.value_counts(normalize=True))

Treino: (8000, 8)
Teste: (2000, 8)

Proporção da classe alvo no treino:
falha_maquina
0    0.966125
1    0.033875
Name: proportion, dtype: float64


## 5. Pipeline com ColumnTransformer

O `ColumnTransformer` aplica tratamento adequado por tipo de variável: escala as numéricas e codifica a variável categórica `tipo`. O SMOTE é aplicado dentro do pipeline e apenas no treino.

In [6]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

preprocessador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="drop"
)

numeric_features, categorical_features

(['temperatura_ar_k',
  'temperatura_processo_k',
  'velocidade_rotacao_rpm',
  'torque_nm',
  'desgaste_ferramenta_min',
  'potencia_operacional',
  'delta_temperatura'],
 ['tipo'])

## 6. Modelos avaliados

A métrica principal passa a ser o `F1-score` da classe de falha, com atenção especial ao `recall`, porque deixar de detectar uma falha pode gerar parada de linha, custo operacional e risco produtivo.

Além de Random Forest e Gradient Boosting, são avaliadas as mesmas três configurações de LightGBM utilizadas no notebook principal. Elas variam `n_estimators`, `learning_rate` e `num_leaves` para comparar capacidade de aprendizado e complexidade. O XGBoost continua opcional quando disponível no ambiente.


In [7]:
configuracoes_lightgbm = [
    {"n_estimators": 100, "learning_rate": 0.05, "num_leaves": 15},
    {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 31},
    {"n_estimators": 300, "learning_rate": 0.03, "num_leaves": 31},
]

modelos = {
    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=80,
        learning_rate=0.06,
        max_depth=2,
        random_state=42
    )
}

for configuracao in configuracoes_lightgbm:
    nome = (
        "LightGBM "
        f"(n={configuracao['n_estimators']}, "
        f"lr={configuracao['learning_rate']}, "
        f"leaves={configuracao['num_leaves']})"
    )
    modelos[nome] = LGBMClassifier(
        **configuracao,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )

# XGBoost é opcional. Se estiver instalado, será incluído automaticamente.
try:
    from xgboost import XGBClassifier

    negativos = (y_train == 0).sum()
    positivos = (y_train == 1).sum()

    modelos["XGBoost"] = XGBClassifier(
        n_estimators=80,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        scale_pos_weight=negativos / max(positivos, 1),
        random_state=42,
        n_jobs=-1
    )
except Exception as erro:
    print("XGBoost não disponível neste ambiente:", erro)

print("Modelos que participarão da comparação:")
for nome in modelos:
    print("-", nome)


XGBoost não disponível neste ambiente: No module named 'xgboost'
Modelos que participarão da comparação:
- Random Forest
- Gradient Boosting
- LightGBM (n=100, lr=0.05, leaves=15)
- LightGBM (n=200, lr=0.05, leaves=31)
- LightGBM (n=300, lr=0.03, leaves=31)


## 7. Treinamento, thresholds e avaliação por recall/F1

Cada modelo, incluindo as três configurações de LightGBM, é treinado dentro do mesmo pipeline para garantir uma comparação justa. O SMOTE é aplicado somente aos dados de treino.

Também são avaliados thresholds diferentes. Em manutenção preditiva, reduzir o threshold pode aumentar o recall da classe de falha, mas também pode aumentar falsos positivos. O melhor resultado é escolhido primeiro pelo F1-score da falha, depois pelo recall e, em caso de empate, pela acurácia.


In [8]:
resultados = []
melhor_modelo = None

thresholds = [0.25, 0.30, 0.35, 0.40, 0.50]

for nome, classificador in modelos.items():
    pipeline = Pipeline(steps=[
        ("preprocessador", preprocessador),
        ("smote", SMOTE(random_state=42, k_neighbors=3)),
        ("modelo", classificador)
    ])

    pipeline.fit(X_train, y_train)

    if hasattr(pipeline, "predict_proba"):
        probabilidades = pipeline.predict_proba(X_test)[:, 1]
    else:
        probabilidades = pipeline.predict(X_test)

    for threshold in thresholds:
        y_pred = (probabilidades >= threshold).astype(int)

        metricas = {
            "modelo": nome,
            "threshold": threshold,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_falha": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
            "recall_falha": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
            "f1_falha": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
        }

        resultados.append(metricas)

        criterio = (metricas["f1_falha"], metricas["recall_falha"], metricas["accuracy"])
        if melhor_modelo is None or criterio > melhor_modelo["criterio"]:
            melhor_modelo = {
                "criterio": criterio,
                "nome": nome,
                "threshold": threshold,
                "pipeline": pipeline,
                "metricas": metricas,
                "y_pred": y_pred
            }

df_resultados = pd.DataFrame(resultados).sort_values(
    by=["f1_falha", "recall_falha", "accuracy"],
    ascending=False
)

df_resultados

,modelo,threshold,accuracy,precision_falha,recall_falha,f1_falha
24,"LightGBM (n=300, lr=0.03, leaves=31)",0.50,0.9780,0.642857,0.794118,0.710526
19,"LightGBM (n=200, lr=0.05, leaves=31)",0.50,0.9780,0.646341,0.779412,0.706667
18,"LightGBM (n=200, lr=0.05, leaves=31)",0.40,0.9720,0.562500,0.794118,0.658537
17,"LightGBM (n=200, lr=0.05, leaves=31)",0.35,0.9705,0.544554,0.808824,0.650888
22,"LightGBM (n=300, lr=0.03, leaves=31)",0.35,0.9670,0.509091,0.823529,0.629213
23,"LightGBM (n=300, lr=0.03, leaves=31)",0.40,0.9680,0.519231,0.794118,0.627907
16,"LightGBM (n=200, lr=0.05, leaves=31)",0.30,0.9665,0.504505,0.823529,0.625698
21,"LightGBM (n=300, lr=0.03, leaves=31)",0.30,0.9625,0.470588,0.823529,0.598930
20,"LightGBM (n=300, lr=0.03, leaves=31)",0.25,0.9605,0.456000,0.838235,0.590674
15,"LightGBM (n=200, lr=0.05, leaves=31)",0.25,0.9610,0.459016,0.823529,0.589474


## 8. Relatório do melhor modelo e comparação do LightGBM

O relatório apresenta o melhor pipeline geral e, separadamente, as melhores linhas obtidas pelas configurações do LightGBM. Assim é possível verificar o desempenho do novo algoritmo mesmo quando outro modelo vence a seleção final.


In [9]:
print("Melhor modelo geral:", melhor_modelo["nome"])
print("Threshold escolhido:", melhor_modelo["threshold"])
print("\nMétricas:")
print(melhor_modelo["metricas"])

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, melhor_modelo["y_pred"]))

print("\nRelatório de classificação:")
print(classification_report(
    y_test,
    melhor_modelo["y_pred"],
    target_names=["normal", "falha"]
))

resultados_lightgbm = df_resultados[
    df_resultados["modelo"].str.startswith("LightGBM")
].copy()

print("\nMelhores resultados do LightGBM:")
display(resultados_lightgbm.head(10))


Melhor modelo geral: LightGBM (n=300, lr=0.03, leaves=31)
Threshold escolhido: 0.5

Métricas:
{'modelo': 'LightGBM (n=300, lr=0.03, leaves=31)', 'threshold': 0.5, 'accuracy': 0.978, 'precision_falha': 0.6428571428571429, 'recall_falha': 0.7941176470588235, 'f1_falha': 0.7105263157894737}

Matriz de confusão:
[[1902   30]
 [  14   54]]

Relatório de classificação:
              precision    recall  f1-score   support

      normal       0.99      0.98      0.99      1932
       falha       0.64      0.79      0.71        68

    accuracy                           0.98      2000
   macro avg       0.82      0.89      0.85      2000
weighted avg       0.98      0.98      0.98      2000


Melhores resultados do LightGBM:


,modelo,threshold,accuracy,precision_falha,recall_falha,f1_falha
24,"LightGBM (n=300, lr=0.03, leaves=31)",0.50,0.9780,0.642857,0.794118,0.710526
19,"LightGBM (n=200, lr=0.05, leaves=31)",0.50,0.9780,0.646341,0.779412,0.706667
18,"LightGBM (n=200, lr=0.05, leaves=31)",0.40,0.9720,0.562500,0.794118,0.658537
17,"LightGBM (n=200, lr=0.05, leaves=31)",0.35,0.9705,0.544554,0.808824,0.650888
22,"LightGBM (n=300, lr=0.03, leaves=31)",0.35,0.9670,0.509091,0.823529,0.629213
23,"LightGBM (n=300, lr=0.03, leaves=31)",0.40,0.9680,0.519231,0.794118,0.627907
16,"LightGBM (n=200, lr=0.05, leaves=31)",0.30,0.9665,0.504505,0.823529,0.625698
21,"LightGBM (n=300, lr=0.03, leaves=31)",0.30,0.9625,0.470588,0.823529,0.598930
20,"LightGBM (n=300, lr=0.03, leaves=31)",0.25,0.9605,0.456000,0.838235,0.590674
15,"LightGBM (n=200, lr=0.05, leaves=31)",0.25,0.9610,0.459016,0.823529,0.589474


## 9. Salvamento do modelo final com joblib

O arquivo salvo contém o melhor pipeline geral, o threshold escolhido, a lista de features e as métricas. Assim, o dashboard consegue carregar o modelo e aplicar a mesma preparação automaticamente.

A comparação completa, incluindo todas as configurações e thresholds do LightGBM, também é exportada em CSV.


In [10]:
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

artefato_modelo = {
    "pipeline": melhor_modelo["pipeline"],
    "modelo_nome": melhor_modelo["nome"],
    "threshold": melhor_modelo["threshold"],
    "features": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "metrics": melhor_modelo["metricas"],
    "configuracoes_lightgbm": configuracoes_lightgbm
}

caminho_modelo = MODELS_DIR / "modelo_final_manutencao_preditiva_lightgbm.joblib"
caminho_comparacao = MODELS_DIR / "comparacao_modelos_complementar_lightgbm.csv"

joblib.dump(artefato_modelo, caminho_modelo)
df_resultados.to_csv(caminho_comparacao, index=False)

print(f"Modelo salvo em: {caminho_modelo}")
print(f"Comparação salva em: {caminho_comparacao}")


Modelo salvo em: models\modelo_final_manutencao_preditiva_lightgbm.joblib
Comparação salva em: models\comparacao_modelos_complementar_lightgbm.csv


## 10. Como usar no dashboard

O dashboard complementar está em `dashboard/dashboard_monitoramento.py`.

Para executar localmente:

```bash
streamlit run dashboard/dashboard_monitoramento.py
```

O notebook salva o melhor pipeline geral no arquivo `models/modelo_final_manutencao_preditiva_lightgbm.joblib`. Caso o dashboard ainda procure o nome antigo do artefato, atualize nele o caminho de carregamento para esse novo arquivo.

O artefato registra também o nome do algoritmo vencedor e as configurações de LightGBM avaliadas. O dashboard pode carregar o pipeline, receber um novo CSV e gerar a probabilidade de falha por equipamento usando o threshold selecionado.
